# KASA-42 — Colab rehearsal

**Purpose: break things here so they cannot break on the H200.**

Runs the *same* `src/kasa42/` code Thursday will run. Only the numbers differ
(fewer languages, smaller batches, fewer steps). Every bug found here is a bug
that would otherwise eat part of a 48-hour window.

**The reason for Colab over Kaggle: Google Drive.** Colab runtimes disconnect,
and `/content` is wiped when they do. Mounting Drive means the manifest — the
one genuinely slow step — writes its per-config parts somewhere permanent. A
disconnect then costs one config instead of the whole run.

**Runtime → Change runtime type → T4 GPU** before starting.

| | Colab T4 | H200 |
|---|---|---|
| Precision | fp16 + GradScaler | bf16 (native) |
| VRAM | 16 GB | 141 GB |
| `batch_duration` | 60 | 320+ |
| Gradient checkpointing | required | unnecessary |

A T4 is Turing (sm_75) and has **no native bf16**. `torch.cuda.is_bf16_supported()`
returns True anyway because it counts software emulation — which measured
4.0 audio-seconds/second, roughly 10× slower than fp16. `has_native_bf16()`
checks compute capability instead.

## 0 · Mount Drive

Skip only if you accept losing manifest progress on every disconnect.

In [ ]:
USE_DRIVE = True

import pathlib

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = pathlib.Path('/content/drive/MyDrive/kasa42')
else:
    PERSIST = pathlib.Path('/content/kasa42_persist')

PERSIST.mkdir(parents=True, exist_ok=True)
(PERSIST / 'manifest_parts').mkdir(exist_ok=True)
print('persisting to', PERSIST)
print('already done:', len(list((PERSIST / 'manifest_parts').glob('*.parquet'))), '/ 42 configs')

## 1 · Code and dependencies

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = 'https://github.com/NasamuAlhassan/kasa42.git'
WORK = pathlib.Path('/content/kasa42')

if WORK.exists():
    subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)], check=True)

os.chdir(WORK)
if str(WORK / 'src') not in sys.path:
    sys.path.insert(0, str(WORK / 'src'))

# Drop stale modules: after a git pull, the runtime would otherwise execute old
# bytecode while tracebacks render the new source — baffling and slow to debug.
for name in [m for m in sys.modules if m.startswith('kasa42')]:
    del sys.modules[name]

rev = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip()
print('cwd', os.getcwd(), '| rev', rev)

In [ ]:
!pip install -q -U transformers datasets soundfile librosa jiwer onnx onnxruntime 2>&1 | tail -2

import os
# HF's Xet CAS backend fails on large files; plain HTTP is slower but reliable.
os.environ['HF_HUB_DISABLE_XET'] = '1'

import torch, transformers
sys.path.insert(0, 'src')
from kasa42.asr.train import has_native_bf16

print('torch', torch.__version__, '| transformers', transformers.__version__)
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    print(f'{torch.cuda.get_device_name(0)}  sm_{cc[0]}{cc[1]}  '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
    print(f'native bf16: {has_native_bf16()}  '
          f'(is_bf16_supported() says {torch.cuda.is_bf16_supported()} — counts emulation)')
else:
    print('NO GPU — Runtime > Change runtime type > T4 GPU')

## 2 · Manifest — the critical path

Metadata from all 533 shards. Network-bound, so minutes here versus hours on a
domestic connection. Parts are written **to Drive**, so this cell is safe to
re-run after any disconnect — finished configs are skipped.

6 workers, not 16: each buffers ~4 MB blocks, and 16 concurrent readers against
1.5 GB shards exhausted a 16 GB kernel.

In [ ]:
PARTS = str(PERSIST / 'manifest_parts')

!HF_HUB_DISABLE_XET=1 python -m kasa42.data.build_manifest \
    --workers 6 --parts-dir "$PARTS" --out results/manifest.parquet

In [ ]:
# Progress check — safe to run any time, including after a disconnect.
import pathlib
parts = sorted(pathlib.Path(PARTS).glob('*.parquet'))
print(f'{len(parts)}/42 configs')
for p in parts:
    print(f'  {p.stem:28s} {p.stat().st_size/1e6:7.2f} MB')
if len(parts) < 42:
    print('\nRe-run the cell above — completed configs are skipped.')

## 3 · Splits, vocab, mixture — the first real numbers

In [ ]:
!python -m kasa42.data.splits
!python -m kasa42.data.vocab
!python -m kasa42.data.mixture --alpha 0.5 --cap-hours 40 --budget-hours 700

In [ ]:
# Copy the four frozen inputs to Drive, then commit them to the repo.
# Once these exist, Thursday starts with data prep already done.
import shutil, pathlib
need = ['results/manifest.parquet', 'results/splits.json',
        'results/vocab.json', 'results/mixture.json']
missing = [f for f in need if not pathlib.Path(f).exists()]
if missing:
    print('MISSING:', missing)
else:
    for f in need:
        shutil.copy(f, PERSIST / pathlib.Path(f).name)
        print(f'ok  {f:32s} {pathlib.Path(f).stat().st_size/1e6:8.2f} MB  -> Drive')
    print('\nDownload these from Drive and commit them to the repo.')

## 4 · Logic checks

In [ ]:
!python tests/test_text.py

In [ ]:
!python tests/test_pipeline.py

## 5 · Benchmark → H200 estimate

Throughput in **seconds of audio per second of wall clock**, which scales across
GPUs far better than step counts. Also prints the optimiser/activation memory
split — AdamW costs ~9.7 GB for a 606M model before a single activation, which
is why anything above `batch_duration=60` OOMs on 16 GB.

In [ ]:
!python tests/bench_gpu.py --minutes 3 --batch-durations 30 60 90

## 6 · A real (tiny) training run

In [ ]:
SMOKE_LANGS = ['Kusaal_kus', 'Asante_Twi_twi', 'Ewe_ewe', 'Dagaare_dga', 'Mampruli_maw']

from huggingface_hub import snapshot_download
snapshot_download('ghananlpcommunity/ghana-speech', repo_type='dataset',
                  local_dir='data/parquet',
                  allow_patterns=[f'{c}/train-00000-*' for c in SMOKE_LANGS],
                  max_workers=6)
!du -sh data/parquet

In [ ]:
from kasa42.asr.train import train, TrainConfig

# T4 settings. The H200 needs neither of these.
ckpt = train(TrainConfig(smoke=True, languages=SMOKE_LANGS,
                         batch_duration=60.0,
                         gradient_checkpointing=True,
                         out_dir='checkpoints/smoke'))
print('checkpoint:', ckpt)

## 7 · Export — the demo must outlive the GPU

In [ ]:
!python -m kasa42.asr.export --checkpoint checkpoints/smoke/final.pt \
    --model-config checkpoints/smoke/config.json --out-dir export_smoke

# The check that matters: does it run with the GPU hidden?
!CUDA_VISIBLE_DEVICES='' KASA42_EXPORT=export_smoke KASA42_MODE=onnx python -c "\
import sys; sys.path.insert(0,'src'); import numpy as np; \
from kasa42.app.app import Engine; e=Engine('onnx'); \
t,l,c,dt=e.transcribe(16000, np.zeros(32000,dtype=np.float32)); \
print('mode', e.mode, '| lang', l, '|', f'{dt*1000:.0f}ms')"

## Checklist before Thursday

- [ ] 42/42 manifest configs, and the four frozen inputs saved to Drive **and
      committed to the repo**
- [ ] `test_text.py` and `test_pipeline.py` fully green
- [ ] `bench_gpu.py` gives an H200 estimate — remember a T4 figure under fp16
      scales up by roughly 15×
- [ ] `train(smoke=True)` completes and the loss moves
- [ ] ONNX export runs with `CUDA_VISIBLE_DEVICES=''`

Committing the four data files is the one that matters most: it means the H200
window opens with data preparation already behind you.